# FTR001 · 日内 v3 完整研究

本 Notebook 是两个固定 FTR001 版本共用的开发界面。它通过 saltcore 读取行情、构造无泄漏信号、把每个品种当作独立的一手账户运行，写出规范的 `runs/v3_10y` 文件并检查结果。可执行规则始终集中在 `config/factors.json`；本 Notebook 不建立第二套参数。

In [ ]:
import pandas as pd
from infra.config import PROJECT, factor_strategies, product_ids
from infra.runner import DEFAULT_END, DEFAULT_START, DEFAULT_WARMUP, run_factor

FACTOR_ID = 'FTR001'
RUN_DIR = PROJECT / FACTOR_ID.lower() / 'runs' / 'v3_10y'
PRODUCTS = product_ids()

## 1. 冻结参数
在读取任何数据之前，先检查确切的机器可读配置。

In [ ]:
pd.DataFrame(factor_strategies(FACTOR_ID)).T

## 2. 完整十年运行
预热期和缺口校准都在报告区间之前完成。SC 和 SA 只保留各自真实可得的历史。

In [ ]:
summary = run_factor(
    FACTOR_ID, PRODUCTS,
    warmup_start=DEFAULT_WARMUP, start=DEFAULT_START, end_exclusive=DEFAULT_END,
)
summary

## 3. 单品种 PnL
每条曲线从零开始，代表一手合约。这是金额曲线，不是资金收益率序列。

In [ ]:
strategy_id = factor_strategies(FACTOR_ID)[0]['strategy_id']
product_id = 'SHFE.RB'
pnl = pd.read_csv(RUN_DIR / strategy_id / product_id / 'pnl.csv', parse_dates=['date'])
pnl.set_index('date')['cumulative_net_pnl'].plot(figsize=(13, 4), title=f'{strategy_id} · {product_id}')

## 4. 完整交易台账与退出诊断

In [ ]:
trades = pd.read_csv(RUN_DIR / strategy_id / product_id / 'trades.csv')
display(trades.head())
display(trades.groupby('exit_reason').agg(trades=('trade_id', 'count'), net_pnl=('net_pnl', 'sum')))
display(trades[['net_pnl', 'max_favorable_pnl', 'max_adverse_pnl', 'max_favorable_giveback_ratio']].describe())